In [10]:
import os
import torch
import numpy as np
import argparse
import torch
import torch.nn as nn
import wandb
import polars as pl
import pickle

from utils.general_utils import set_seed
from dvrl.dataset import EssayDataset
from models.features import FeaturesModel
from utils.dvrl_utils import fit_func, pred_func, calc_qwk

In [11]:
for seed in [12, 32, 52]:
    for target_prompt_id in range(1, 9):
        ###################################################
        # Step1. Load Data
        ###################################################
        device = torch.device('cuda')
        set_seed(seed)
        # Load essay data
        print('Loading essay data...')
        dataset = EssayDataset('../data/training_set_rel3.xlsx', '../data/hand_crafted_v3.csv', '../data/readability_features.csv')
        source_data, target_data = dataset.cross_prompt_split(
            target_prompt_set=target_prompt_id,
            add_pos=False,
        )

        model = FeaturesModel().to(device)

        epoch = 500 if target_prompt_id == 8 else 100
        fit_func(
            model,
            np.concatenate([source_data['feature'], source_data['readability']], axis=1),
            source_data['scaled_score'],
            batch_size=512,
            epochs=epoch,
            device=device,
        )

        y_test_pred = pred_func(
            model,
            np.concatenate([target_data['feature'], target_data['readability']], axis=1),
            512,
            device
        )

        # Calculate QWK
        test_qwk = calc_qwk(target_data['scaled_score'], y_test_pred, target_prompt_id, 'score')

        print(f'Test QWK: {test_qwk}')

        import polars as pl
        # y_test_predの値をCSVで保存
        df = pl.DataFrame({
            'essay_id': target_data['essay_id'],
            'y_pred': y_test_pred.flatten()
        })
        df.write_csv(f'../outputs/pseudo_labels/features_model_pred_{target_prompt_id}_seed{seed}.csv')

Loading essay data...
Test QWK: 0.7862664083329126
Loading essay data...
Test QWK: 0.6503877319857105
Loading essay data...
Test QWK: 0.5789279434941037
Loading essay data...
Test QWK: 0.5765522370719522
Loading essay data...
Test QWK: 0.7069426886382422
Loading essay data...
Test QWK: 0.5736661620686458
Loading essay data...
Test QWK: 0.6421380549694337
Loading essay data...
Test QWK: 0.6273773082801699
Loading essay data...
Test QWK: 0.7719391932960218
Loading essay data...
Test QWK: 0.6433781812287243
Loading essay data...
Test QWK: 0.5714217333639147
Loading essay data...
Test QWK: 0.578352543526331
Loading essay data...
Test QWK: 0.7072224843694703
Loading essay data...
Test QWK: 0.5671364410717051
Loading essay data...
Test QWK: 0.6600030363311911
Loading essay data...
Test QWK: 0.6241313319711921
Loading essay data...
Test QWK: 0.7809362417113628
Loading essay data...
Test QWK: 0.6501563995936535
Loading essay data...
Test QWK: 0.5726481807794541
Loading essay data...
Test QWK: 